# Adjust Threshold for Keyframe Extraction

This notebook uses pre-computed embeddings saved in pkl file to extract keyframes with different thresholds without recalculating embeddings.

## Input Data Structure:
```python
embedding = {
    "paths": [
        "L01_V001/0.jpg",
        "L01_V001/7.jpg",
        "L01_V002/5.jpg",
        ...
    ],
    "embeddings": [
        [0.1, 0.2, 0.3, ...],  # embedding for "L01_V001/0.jpg"
        [0.4, 0.5, 0.6, ...],  # embedding for "L01_V001/7.jpg"
        [0.7, 0.8, 0.9, ...],  # embedding for "L01_V002/5.jpg"
        ...
    ]
}
```

## Output Data Structure:
Similar to input but only contains extracted keyframes.

In [25]:
import pickle
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import json
import multiprocessing as mp
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

def process_video_worker(video_name, video_paths, video_embeddings, similarity_threshold):
    """
    Worker function for multiprocessing.
    Processes keyframe extraction for a single video.
    
    Args:
        video_name (str): Video name (e.g., "L01_V001")
        video_paths (list): List of frame paths for the video (e.g., ["L01_V001/0.jpg", "L01_V001/7.jpg"])
        video_embeddings (list): List of embeddings corresponding to paths
        similarity_threshold (float): Similarity threshold to determine keyframes (e.g., 0.85)
    
    Returns:
        tuple: (video_name, keyframe_paths, keyframe_embeddings, n_keyframes, n_total_frames)
    """
    # Sort by frame number
    path_embedding_pairs = list(zip(video_paths, video_embeddings))
    path_embedding_pairs.sort(key=lambda x: int(Path(x[0]).stem))
    
    sorted_paths = [pair[0] for pair in path_embedding_pairs]
    sorted_embeddings = [pair[1] for pair in path_embedding_pairs]
    
    # Extract keyframes
    if len(sorted_embeddings) <= 1:
        keyframe_indices = [0] if len(sorted_embeddings) > 0 else []
    else:
        keyframe_indices = [0]
        last_keyframe_embedding = sorted_embeddings[0]
        
        for i in range(1, len(sorted_embeddings)):
            similarity = cosine_similarity([last_keyframe_embedding], [sorted_embeddings[i]])[0][0]
            if similarity < similarity_threshold:
                keyframe_indices.append(i)
                last_keyframe_embedding = sorted_embeddings[i]
    
    # Return paths and embeddings of keyframes
    keyframe_paths = [sorted_paths[i] for i in keyframe_indices]
    keyframe_embeddings = [sorted_embeddings[i] for i in keyframe_indices]
    
    return video_name, keyframe_paths, keyframe_embeddings, len(keyframe_indices), len(sorted_paths)

In [26]:
class KeyframeAdjuster:
    """
    Class for extracting keyframes with different thresholds from pre-computed embeddings.
    Processes data with format: {"paths": [...], "embeddings": [...]}
    """
    
    def __init__(self, similarity_threshold):
        self.similarity_threshold = similarity_threshold
    
    def _group_by_video(self, data):
        """
        Group paths and embeddings by video_name.
        
        Args:
            data: dict with keys "paths" and "embeddings"
        
        Returns:
            dict: {video_name: {"paths": [...], "embeddings": [...]}}
        """
        video_groups = defaultdict(lambda: {"paths": [], "embeddings": []})
        
        for path, embedding in zip(data["paths"], data["embeddings"]):
            video_name = Path(path).parts[0]  # Extract video name from path
            video_groups[video_name]["paths"].append(path)
            video_groups[video_name]["embeddings"].append(embedding)
        
        return dict(video_groups)
    
    def process_data(self, data, n_workers=None):
        """
        Process data to extract keyframes for all videos using multi-core processing.
        
        Args:
            data: dict with keys "paths" and "embeddings"
            n_workers: Number of workers, defaults to cpu_count - 1
        
        Returns:
            dict: {"paths": [...], "embeddings": [...]} containing only keyframes
        """
        if n_workers is None:
            n_workers = max(1, mp.cpu_count() - 1)
        
        # Group data by video
        video_groups = self._group_by_video(data)
        
        return self._process_multicore(video_groups, n_workers)
    
    def _process_multicore(self, video_groups, n_workers):
        """Process in parallel (multi-core)"""
        all_keyframe_paths = []
        all_keyframe_embeddings = []
        
        # Prepare data for multiprocessing
        video_args = [(video_name, group["paths"], group["embeddings"], self.similarity_threshold) 
                     for video_name, group in video_groups.items()]
        
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            # Submit all tasks with separate arguments
            futures = {executor.submit(process_video_worker, *args): args[0] 
                      for args in video_args}
            
            # Collect results with progress bar
            with tqdm(total=len(futures), desc="Extracting keyframes", disable=False) as pbar:
                for future in as_completed(futures):
                    video_name = futures[future]
                    try:
                        video_name, keyframe_paths, keyframe_embeddings, n_keyframes, n_frames = future.result()
                        
                        # Add results to combined list
                        all_keyframe_paths.extend(keyframe_paths)
                        all_keyframe_embeddings.extend(keyframe_embeddings)
                        
                        pbar.update(1)
                    except Exception as exc:
                        print(f"❌ Video {video_name} encountered error: {exc}")
                        pbar.update(1)
                      
        return {
            "paths": all_keyframe_paths,
            "embeddings": all_keyframe_embeddings
        }

    def save_results(self, keyframe_data, output_file):
        """Save results to JSON or pickle file."""
        # Add length field to the data
        data_with_length = {
            "paths": keyframe_data["paths"],
            "embeddings": keyframe_data["embeddings"],
            "length": len(keyframe_data["paths"])
        }
        
        if output_file.endswith('.json'):
            # Convert numpy arrays to lists for JSON serialization
            json_data = {
                "paths": data_with_length["paths"],
                "embeddings": [emb.tolist() if hasattr(emb, 'tolist') else emb 
                              for emb in data_with_length["embeddings"]],
                "length": data_with_length["length"]
            }
            with open(output_file, 'w') as f:
                json.dump(json_data, f, indent=2)
        elif output_file.endswith('.pkl'):
            with open(output_file, 'wb') as f:
                pickle.dump(data_with_length, f)
        else:
            raise ValueError("Output file must have .json or .pkl extension")

## config and run

In [27]:
CONFIGS = [
    {
        "input_file": "lucifer-e-clip-6.pkl",
        "similarity_threshold": 0.95
    },
]

EMBEDDING_OUTPUT_PATH = "all_keyframes_combined.pkl"

# Run keyframe extraction for multiple configurations
results_summary = []
all_results = {
    "paths": [],
    "embeddings": [],
    "length": 0
}

for config in CONFIGS:
    try:
        # Read data from pkl file
        with open(config['input_file'], 'rb') as f:
            data = pickle.load(f)
        
        # Check data consistency
        if len(data['paths']) != len(data['embeddings']):
            raise ValueError(f"Number of paths ({len(data['paths'])}) and embeddings ({len(data['embeddings'])}) don't match!")
        
        # Initialize adjuster and process
        adjuster = KeyframeAdjuster(similarity_threshold=config['similarity_threshold'])
        keyframe_data = adjuster.process_data(data)
        
        # Extend all_results with current keyframe_data
        all_results["paths"].extend(keyframe_data["paths"])
        all_results["embeddings"].extend(keyframe_data["embeddings"])
        all_results["length"] += len(keyframe_data["paths"])
        
        # Record success
        results_summary.append({
            'config': config,
            'status': 'success',
            'original_frames': len(data['paths']),
            'keyframes_extracted': len(keyframe_data['paths']),
            'reduction_percent': (1 - len(keyframe_data['paths']) / len(data['paths'])) * 100
        })
        
    except FileNotFoundError:
        print(f"❌ Error: File {config['input_file']} not found!")
        results_summary.append({
            'config': config,
            'status': 'failed',
            'error': 'File not found'
        })
    except Exception as e:
        print(f"❌ Error processing {config['input_file']}: {e}")
        results_summary.append({
            'config': config,
            'status': 'failed',
            'error': str(e)
        })

# Save all results to single file
if all_results["length"] > 0:
    adjuster = KeyframeAdjuster(similarity_threshold=0.0)  # threshold doesn't matter for saving
    adjuster.save_results(all_results, EMBEDDING_OUTPUT_PATH)
    print(f"💾 All results saved to {EMBEDDING_OUTPUT_PATH}")

# Show summary results
successful_runs = [r for r in results_summary if r['status'] == 'success']
failed_runs = [r for r in results_summary if r['status'] == 'failed']

print(f"✅ Completed: {len(successful_runs)}/{len(CONFIGS)} configurations")

if successful_runs:
    print(f"\n📊 Successful configurations:")
    for result in successful_runs:
        config = result['config']
        print(f"   📄 {config['input_file']} (threshold: {config['similarity_threshold']})")
        print(f"      📊 {result['original_frames']:,} → {result['keyframes_extracted']:,} frames ({result['reduction_percent']:.1f}% reduction)")

if failed_runs:
    print(f"\n❌ Failed configurations:")
    for result in failed_runs:
        print(f"   📄 {result['config']['input_file']}: {result['error']}")

/tmp/ipykernel_5013/1472890829.py:22: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)
Extracting keyframes: 100%|██████████| 59/59 [00:16<00:00,  3.68it/s]



💾 All results saved to all_keyframes_combined.pkl
✅ Completed: 1/1 configurations

📊 Successful configurations:
   📄 lucifer-e-clip-6.pkl (threshold: 0.95)
      📊 98,823 → 34,665 frames (64.9% reduction)
